# Notebook Goal
```text
silver.rides -> drivers/payment because rides has only one table
    -> OPTIMIZE
    -> ZORDER BY ride_id

silver.driver
    -> Liquid Clustering
    -> CLUSTER BY ride_id
```

First inspect table/files

In [0]:
%sql
DESCRIBE DETAIL
ride_lakehouse_dev.silver.rides;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,db45e9f9-701c-4e1f-bb00-35f46b37f5a1,ride_lakehouse_dev.silver.rides,null,,2026-09-29T18:34:10.280Z,2026-09-29T18:34:36.000Z,List(),List(),1,16026,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, io.unitycatalog.tableId -> 2f38a9e1-8361-4efa-ba98-f35519f59e80, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-ba0e2d87-2e39-41e7-941e-b28647640144, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-c2c1fec5-2f08-4be1-8774-853bceed4c75)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL
ride_lakehouse_dev.silver.drivers;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a74ebd93-0445-469f-9f23-fcf72a307997,ride_lakehouse_dev.silver.drivers,null,,2026-09-29T18:29:19.724Z,2026-09-30T17:05:18.000Z,List(),List(),11,43319,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, io.unitycatalog.tableId -> b1e6dd8d-38df-4b17-9729-f81080334637, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-f0d31688-ed4e-45be-947e-5fd3be639daa, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8583f7bd-0816-444c-838d-e69571621cec)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE DETAIL
ride_lakehouse_dev.silver.payments;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a9c93ca5-7e22-4e46-9b86-79174e43b74f,ride_lakehouse_dev.silver.payments,null,,2026-09-29T18:35:19.027Z,2026-09-29T18:35:43.000Z,List(),List(),7,30483,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, io.unitycatalog.tableId -> 7b8ca812-02bc-47ef-a5d5-1edd56d05186, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-2d6b2f5b-ec39-4755-86c6-37e564fc5e7c, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d1d18576-45f0-4450-b0ba-029164933f59)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


As a result,  rides table has only one table but drivers has 11 file and total 43kb also payments has 7 files with 30kb which means they are so small filess.

I was planning to use optimize on rides but there is only one file already, so I will apply this on drivers


In [0]:
%sql
OPTIMIZE
ride_lakehouse_dev.silver.drivers;

path,metrics
,"List(1, 11, List(6409, 6409, 6409.0, 1, 6409), List(3742, 5675, 3938.090909090909, 11, 43319), 0, null, null, 0, 1, 11, 0, true, 0, 0, 1790793457046, 1790793469569, 8, 1, null, List(0, 0), null, 11, 11, 6648, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE DETAIL
ride_lakehouse_dev.silver.drivers;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,a74ebd93-0445-469f-9f23-fcf72a307997,ride_lakehouse_dev.silver.drivers,null,,2026-09-29T18:29:19.724Z,2026-09-30T18:37:49.000Z,List(),List(),1,6409,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, io.unitycatalog.tableId -> b1e6dd8d-38df-4b17-9729-f81080334637, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-f0d31688-ed4e-45be-947e-5fd3be639daa, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8583f7bd-0816-444c-838d-e69571621cec)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


now we have only one file for drivers and it will be quicker to find data because spark doesn't need to open all 11 files adn read metadata to find data.

In [0]:
%sql
DESCRIBE HISTORY
ride_lakehouse_dev.silver.drivers;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-09-30T18:37:49.000Z,77111473424338,dogancandemir2001@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1647628881007744),3fd7056d-8b80-4ad9-b584-35b3a4a31350,0930-183213-tg7gclr7-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 11, numRemovedBytes -> 43319, p25FileSize -> 6409, numDeletionVectorsRemoved -> 0, minFileSize -> 6409, numAddedFiles -> 1, maxFileSize -> 6409, p75FileSize -> 6409, p50FileSize -> 6409, numAddedBytes -> 6409)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
4,2026-09-30T17:05:18.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 3bd6ffb2-b325-40e1-aa4f-db6ba327d128, epochId -> 1, statsOnLoad -> false)",null,List(3795482916102374),f8a251d9-d5ad-4d9d-945f-3cb467b444f1,0930-165855-ca2uj070-v2n,3,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 10, numOutputBytes -> 37644, numAddedFiles -> 10)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
3,2026-09-30T17:04:46.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 3bd6ffb2-b325-40e1-aa4f-db6ba327d128, epochId -> -1, statsOnLoad -> false)",null,List(3795482916102374),f8a251d9-d5ad-4d9d-945f-3cb467b444f1,0930-165855-ca2uj070-v2n,2,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-29T18:30:01.000Z,77111473424338,dogancandemir2001@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4361174249883579),f22f236c-a837-4b37-ab63-df1135e7a9c8,0929-181018-n22nnjgu-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 36, numRemovedBytes -> 125860, p25FileSize -> 5675, numDeletionVectorsRemoved -> 0, minFileSize -> 5675, numAddedFiles -> 1, maxFileSize -> 5675, p75FileSize -> 5675, p50FileSize -> 5675, numAddedBytes -> 5675)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-29T18:29:57.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 3bd6ffb2-b325-40e1-aa4f-db6ba327d128, epochId -> 0, statsOnLoad -> true)",null,List(4361174249883579),f22f236c-a837-4b37-ab63-df1135e7a9c8,0929-181018-n22nnjgu-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 40, numOutputBytes -> 125860, numAddedFiles -> 36)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-29T18:29:21.000Z,77111473424338,dogancandemir2001@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""b1e6dd8d-38df-4b17-9729-f81080334637"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f0d31688-ed4e-45be-947e-5fd3be639daa"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-8583f7bd-0816-444c-838d-e69571621cec""}, statsOnLoad -> false)",null,List(4361174249883579),f22f236c-a837-4b37-ab63-df1135e7a9c8,0929-181018-n22nnjgu-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
%sql
DESCRIBE HISTORY
ride_lakehouse_dev.silver.rides;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-29T18:34:36.000Z,77111473424338,dogancandemir2001@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4361174249883579),c7eabf05-3d70-49e1-bfbc-1937c75fbbd7,0929-181018-n22nnjgu-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 120, numRemovedBytes -> 553045, p25FileSize -> 16026, numDeletionVectorsRemoved -> 0, minFileSize -> 16026, numAddedFiles -> 1, maxFileSize -> 16026, p75FileSize -> 16026, p50FileSize -> 16026, numAddedBytes -> 16026)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
1,2026-09-29T18:34:30.000Z,77111473424338,dogancandemir2001@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 3e934fc3-0fd9-49da-9726-a4b7ec674c33, epochId -> 0, statsOnLoad -> true)",null,List(4361174249883579),c7eabf05-3d70-49e1-bfbc-1937c75fbbd7,0929-181018-n22nnjgu-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 200, numOutputBytes -> 553045, numAddedFiles -> 120)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-29T18:34:11.000Z,77111473424338,dogancandemir2001@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""io.unitycatalog.tableId"":""2f38a9e1-8361-4efa-ba98-f35519f59e80"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ba0e2d87-2e39-41e7-941e-b28647640144"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-c2c1fec5-2f08-4be1-8774-853bceed4c75""}, statsOnLoad -> false)",null,List(4361174249883579),c7eabf05-3d70-49e1-bfbc-1937c75fbbd7,0929-181018-n22nnjgu-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


I guess Databricks apply OPTIMZIE  by ıtself because I didn't so thats why rides is only 1 file

# Z-ordering

In [0]:
payment_schema = (
    spark.table(
        "ride_lakehouse_dev.silver.payments"
    ).schema
)

print(payment_schema)

StructType([StructField('amount', DoubleType(), True), StructField('generated_at', TimestampType(), True), StructField('payment_id', StringType(), True), StructField('payment_method', StringType(), True), StructField('payment_timestamp', TimestampType(), True), StructField('ride_id', StringType(), True), StructField('_rescued_data', StringType(), True), StructField('_ingested_at', TimestampType(), True), StructField('_source_file', StringType(), True), StructField('_silver_processed_at', TimestampType(), True)])


In [0]:
%sql
OPTIMIZE
ride_lakehouse_dev.silver.payments

ZORDER BY (ride_id);

path,metrics
,"List(1, 7, List(8203, 8203, 8203.0, 1, 8203), List(3638, 7974, 4354.714285714285, 7, 30483), 0, List(minCubeSize(107374182400), List(0, 0), List(7, 30483), 0, List(7, 30483), 1, null), null, 0, 1, 7, 0, false, 0, 0, 1790794343123, 1790794351190, 8, 1, null, List(0, 0), null, 10, 10, 537, 0, null, null, 0)"


Now, we re-order the payments table/files by rides id, so spark can find ayment row by rieds id without scaning all table by using meta date min/max

# Liquid clustering
The diferneces between z-ordering is that liquid clustering can evolve
- Liquid Clustering and Z-Ordering aren't compatible on the same table.
So I will not apply liquid clustering because mistakenly I applyed z ordering on payment nad payment table is the only table taht have multiple file. If I want I can restore the table but or time travel. 

- There is no reson to apply liquid clustering on one file table

In [0]:
%sql
ALTER TABLE
ride_lakehouse_dev.silver.drivers

CLUSTER BY (ride_id);